# 03. SAST for General Code

## Background

Static Application Security Testing (SAST) analyzes source code without executing it, finding vulnerabilities at development time. Semgrep is the community standard: it uses lightweight pattern rules on an AST representation, runs in milliseconds per file, and supports custom rules without complex setup.

## Why This Matters

Automated security scanning in CI prevents vulnerabilities from reaching production. Each scan type catches a different class of vulnerability — layering them provides defense in depth.

In [ ]:
import ast, re, json
from dataclasses import dataclass
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print(f'03. SAST for General Code ready')


## Semgrep-style AST Pattern Matching

In [ ]:
import ast, re
from dataclasses import dataclass
from typing import Optional

@dataclass
class SASTFinding:
    rule_id: str
    severity: str
    message:  str
    file:     str
    line:     int
    fix:      str = ''


class ASTScanner:
    def __init__(self):
        self._rules = []

    def add_rule(self, rule_id, severity, message, checker_fn, fix=''):
        self._rules.append((rule_id, severity, message, checker_fn, fix))

    def scan_code(self, code: str, filename='<code>') -> list:
        findings = []
        try:
            tree = ast.parse(code)
        except SyntaxError as e:
            return [SASTFinding('PARSE_ERR','ERROR',str(e),filename,0)]
        for rule_id, severity, message, checker, fix in self._rules:
            for node in ast.walk(tree):
                result = checker(node)
                if result:
                    findings.append(SASTFinding(rule_id, severity, message,
                                                filename, getattr(node,'lineno',0), fix))
        return findings


scanner = ASTScanner()

# Rule 1: eval() usage
scanner.add_rule('S001','CRITICAL','eval() allows arbitrary code execution',
    lambda n: isinstance(n, ast.Call) and isinstance(getattr(n,'func',None), ast.Name) and n.func.id == 'eval',
    'Replace with ast.literal_eval() for data parsing')

# Rule 2: subprocess shell=True
def check_shell_true(node):
    if not isinstance(node, ast.Call): return False
    for kw in getattr(node, 'keywords', []):
        if kw.arg == 'shell' and isinstance(kw.value, ast.Constant) and kw.value.value is True:
            return True
    return False
scanner.add_rule('S002','HIGH','subprocess with shell=True enables command injection',
    check_shell_true, 'Use list form: subprocess.run(["cmd", arg])')

# Rule 3: pickle.loads
scanner.add_rule('S003','HIGH','pickle.loads() executes arbitrary code',
    lambda n: (isinstance(n, ast.Call) and isinstance(getattr(n,'func',None), ast.Attribute)
               and n.func.attr == 'loads'
               and isinstance(n.func.value, ast.Name) and n.func.value.id == 'pickle'),
    'Use json.loads() for data interchange')

VULN_CODE = """
import subprocess, pickle, os

def run(cmd):
    result = eval(cmd)  # dangerous
    subprocess.run(cmd, shell=True)  # dangerous
    return result

def load(data):
    return pickle.loads(data)  # dangerous
"""

findings = scanner.scan_code(VULN_CODE, 'app.py')
for f in findings:
    print(f'[{f.severity}] {f.rule_id} line {f.line}: {f.message}')
    print(f'  Fix: {f.fix}')


## Custom Rules for ML Pipelines

In [ ]:
ML_RULES = [
    ('ML001', 'HIGH',   'yaml.load() without Loader executes arbitrary code',
     r"yaml\.load\s*\([^,)]+\)(?!.*Loader)",
     'Use yaml.safe_load() or yaml.load(data, Loader=yaml.SafeLoader)'),
    ('ML002', 'MEDIUM', 'Unpickled model from untrusted source',
     r"joblib\.load|pickle\.load|torch\.load",
     'Only load models from trusted sources; use model hashing to verify integrity'),
    ('ML003', 'MEDIUM', 'Hardcoded model path — breaks reproducibility',
     r"[\'\"]/[a-z]+/[a-z]+.*\.pkl[\'\"]",
     'Use config-driven paths or artifact registries'),
    ('ML004', 'HIGH',   'User input directly in SQL query',
     r"execute\s*\(\s*['\"]SELECT.*%s",
     'Use parameterized queries: cursor.execute(sql, (param,))'),
]

def regex_scan(code, rules, filename='code.py'):
    findings = []
    for lineno, line in enumerate(code.split('\n'), 1):
        for rule_id, severity, message, pattern, fix in rules:
            if re.search(pattern, line):
                findings.append(SASTFinding(rule_id, severity, message, filename, lineno, fix))
    return findings

ML_CODE = """
import yaml, joblib, pickle

model = joblib.load('/models/production/classifier.pkl')
config = yaml.load(open('config.yaml'))
cursor.execute('SELECT * FROM users WHERE id=%s' % user_id)
"""

findings2 = regex_scan(ML_CODE, ML_RULES, 'ml_pipeline.py')
for f in findings2:
    print(f'[{f.severity}] {f.rule_id} line {f.line}: {f.message}')
    print(f'  Fix: {f.fix}')
